# NSR 전사 서버 — 구글 콜랩판 (WhisperX)

폰 대신 콜랩의 GPU가 전사합니다. WhisperX 배치 추론이라 3시간 기록도 몇 분입니다.

**여기서 할 일은 버튼 두 번뿐입니다**

1. 위 메뉴 **런타임 → 모두 실행**. (처음 한 번만: 런타임 → 런타임 유형 변경에서 **T4 GPU** 확인)
2. 마지막 셀에 뜨는 **'NSR 앱에 연결' 버튼**을 누르면 끝 — 주소가 앱에 **자동 저장**됩니다.
   - 콜랩을 컴퓨터로 보고 있다면: 폰 카메라로 **QR** 을 찍고, 열리는 페이지의 버튼을 누르십시오.
   - 버튼·QR 이 안 되면 그때만 주소를 복사해 앱의 설정 → 전사 주소 칸에 붙여넣습니다.

**설정은 전부 앱에서 합니다.** 전사 모델도, 화자 분리도, 화자 분리에 쓰는 허깅페이스
토큰도 앱의 **설정 → 전사** 화면에서 정하고, 전사 요청에 함께 실려 옵니다.
이 노트에서는 고를 것도 저장할 것도 없습니다 — 셀 내용을 건드리지 마십시오.

**빨간 ERROR 가 떠도 놀라지 마십시오** — 설치 셀 끝에 나오는
`pip's dependency resolver ...` (gradio·numba·google-adk·diffusers) 는 콜랩에 미리 깔린
**다른** 프로그램들 이야기입니다. 전사 서버와 무관하고, 실제로 그대로 정상 동작합니다.

이 판은 WhisperX 파이프라인입니다:
- **배치 추론** — 무음 기준으로 잘라 GPU에 묶음으로 넣어 수십 배 빠릅니다.
- **강제 정렬** — 한국어 음소 정렬로 문장·단어 시각이 실측이 됩니다. 문장을 누르면 정확히 그 지점부터 재생됩니다.
- **단어 단위 화자 할당** — 문장 중간에 화자가 바뀌어도 경계가 맞습니다.

**알고 쓰십시오**

- 기록 음성 원본이 구글(콜랩) 서버와 Cloudflare 터널을 지나갑니다. 이 경로가 싫으면 앱의 '내 컴퓨터' 모드를 쓰십시오.
- 주소 끝에 무작위 비밀 문자열이 붙어 있어 주소를 통째로 모르는 남은 못 씁니다. 그래도 주소를 다른 곳에 붙여넣지 마십시오.
- **디스크 걱정은 안 해도 됩니다** — 여기 보이는 디스크는 콜랩이 세션마다 빌려주는 임시 공간이라
  세션이 끝나면 통째로 초기화되고, **구글 드라이브 용량을 먹지 않습니다.** 매 실행마다 설치를
  다시 하는 것도 그 때문입니다(마지막 셀이 사용량을 보여줍니다).
- 콜랩 화면에 '런타임 연결이 끊겼습니다'가 떠도 전사는 계속되고 있을 수 있습니다 — 폰의 진행률이
  움직이면 서버는 살아 있습니다. 전사가 도는 동안 '모두 실행'을 다시 하지는 마십시오. 세션이 정말
  회수되더라도 앱은 받은 부분을 저장해 두고, 기록은 다시 전사할 수 있게 남습니다.
- 세션이 꺼졌으면 '모두 실행'을 다시 — 주소가 새로 나오니 연결 버튼(또는 QR)도 다시 누릅니다.
- 전사할 때만 켜는 개인용입니다. 상시 서버로 두는 것은 콜랩 이용 규칙과 맞지 않습니다.
- 이 노트가 고쳐지면 앱의 '콜랩 노트 열기' 버튼으로 새로 열어야 최신판입니다. 드라이브 사본은 옛 판입니다.


In [ ]:
# 필요한 것 설치 + 터널 프로그램 받기 (3~8분)
#
# whisperx 가 faster-whisper·pyannote.audio·정렬 모델 도구를 함께 끌고 온다.
# nvidia-cudnn/cublas 를 같이 까는 이유: 콜랩 기본 환경의 cuDNN 판이
# ctranslate2 와 어긋나면 첫 전사에서 파이썬이 통째로 죽는다("kernel
# restarted") — 실사용에서 그대로 재현된 사고다.
#
# 왜 진행 상황을 한 줄씩 찍는가
#   pip 을 조용히(-q) 돌리면 몇 분 동안 아무것도 안 찍혀 멈춘 것처럼 보인다.
#   실제로 그렇게 보고됐다. 그래서 pip 출력을 읽어 큰 걸음만 시각과 함께 찍는다.
#
# 왜 numpy 판을 고정하지 않는가
#   콜랩 커널은 시작할 때 이미 numpy 를 실어 둔다. 설치 도중 pip 이 numpy 를
#   갈아치우면 커널 안의 옛 numpy 와 디스크의 새 파일이 어긋나
#     ImportError: cannot import name '_slice' from 'numpy._core.umath'
#   로 터진다. 처음에는 numpy 를 지금 판으로 못박아 막았는데, 그 고정이
#   pip 의 의존성 해결을 헤매게 만들어 설치가 10분 넘게 늘어졌다.
#   그래서 고정은 걷어내고, 판이 바뀌면 커널에 실린 옛 numpy 를 비우는 쪽만 남겼다.
import os
import subprocess
import sys
import time

print("설치를 시작합니다. 큰 걸음마다 아래에 한 줄씩 찍힙니다 (보통 3~8분).")
print("※ 중간에 빨간 ERROR(gradio·numba·google-adk·diffusers ...)가 떠도 고장이 아닙니다.")
print("   콜랩에 미리 깔린 다른 프로그램들 얘기라 전사 서버와 무관합니다.")
print("※ torch 를 새로 받는 판이면 이 셀이 가장 오래 걸립니다. 그대로 두십시오.")

PKGS = ["whisperx", "fastapi", "uvicorn", "python-multipart", "qrcode",
        "nvidia-cudnn-cu12", "nvidia-cublas-cu12"]

# 이 앞머리로 시작하는 줄만 골라 찍는다. 나머지는 진행 막대라 화면만 어지럽힌다.
STEPS = ("Collecting ", "Downloading ", "Installing collected packages",
         "Successfully installed", "ERROR:")


def pip_install(pkgs):
    started = time.time()
    proc = subprocess.Popen(
        [sys.executable, "-m", "pip", "install", *pkgs],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    last = 0.0
    assert proc.stdout is not None
    for line in proc.stdout:
        line = line.rstrip()
        if not line.startswith(STEPS):
            continue
        now = time.time()
        # 큰 파일을 받기 시작하는 줄과 마무리 줄은 언제나 보여준다 — 제일 오래
        # 걸리는 구간(torch 수백 MB)이 눌려 버리면 또 멈춘 것처럼 보인다.
        loud = line.startswith(("Installing collected", "Successfully", "ERROR:")) or (
            line.startswith("Downloading") and " MB" in line
        )
        # 나머지는 2초에 한 줄로 눌러 둔다 — 살아 있다는 것만 보이면 된다.
        if now - last < 2 and not loud:
            continue
        last = now
        print(f"  [{int(now - started)}초] {line[:110]}", flush=True)
    return proc.wait()


def installed_numpy():
    r = subprocess.run(
        [sys.executable, "-c", "import importlib.metadata as m; print(m.version('numpy'))"],
        capture_output=True, text=True,
    )
    return r.stdout.strip() if r.returncode == 0 else ""


before = getattr(sys.modules.get("numpy"), "__version__", "")
if pip_install(PKGS) != 0:
    raise SystemExit("설치가 실패했습니다. 위의 오류 내용을 그대로 개발자에게 보내 주십시오.")

!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
# 내려받기 캐시는 이 세션에서 다시 쓸 일이 없다 — 디스크만 차지하니 비운다.
!pip cache purge > /dev/null 2>&1 || true

# 1) 디스크의 numpy 파일이 서로 섞이지 않았는지 먼저 본다(커널과 무관한 문제).
CHECK = (
    "import numpy\n"
    "if numpy.__version__[:1] >= '2':\n"
    "    import numpy.strings\n"
    "numpy.zeros(3).sum()\n"
)
if subprocess.run([sys.executable, "-c", CHECK], capture_output=True).returncode != 0:
    print("numpy 파일이 섞여 있어 다시 맞춥니다 (30초)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall",
                    "--no-deps", "numpy"], check=False)

# 2) 커널이 이미 실어 둔 numpy 와 디스크의 numpy 가 다른가.
#
#    다르면 뒤 셀이 터진다. 늦게 불러오는 하위 모듈만 새 파일에서 읽혀
#    옛 모듈과 어긋나기 때문이다(ImportError: cannot import name '_slice' ...).
#    한때 sys.modules 에서 numpy 를 비우고 다시 읽게 해 봤는데, numpy 는
#    재적재를 지원하지 않아 이번엔 다른 데서 터졌다
#    (AttributeError: 'numpy.ufunc' object has no attribute '__module__').
#    그래서 정공법으로 간다 — 세션을 한 번 다시 시작한다.
after = installed_numpy()
RESTART_MARK = "/content/.nsr-numpy-restarted"
if before and after and after != before:
    if os.path.exists(RESTART_MARK):
        # 이미 한 번 다시 시작했는데 또 다르다 — 무한 재시작은 만들지 않는다.
        print(f"numpy 가 여전히 어긋납니다({before} vs {after}). 그대로 진행해 봅니다.")
        print("뒤 셀에서 numpy 오류가 나면 런타임 → 세션 다시 시작 후 '모두 실행' 을 해 주십시오.")
    else:
        with open(RESTART_MARK, "w") as f:
            f.write(after)
        print("=" * 62)
        print(f"numpy 가 {before} → {after} 로 바뀌었습니다.")
        print("커널에 실린 옛 numpy 와 섞이면 뒤 셀이 터지므로 세션을 한 번 다시 시작합니다.")
        print("이건 고장이 아니라 정상 절차입니다.")
        print("")
        print("  >>> 잠시 뒤 위 메뉴 [런타임 → 모두 실행] 을 한 번만 더 눌러 주십시오. <<<")
        print("")
        print("두 번째 실행은 이미 다 깔려 있어 이 셀을 20~40초면 지나갑니다.")
        print("=" * 62)
        time.sleep(3)  # 위 안내가 화면에 남도록 잠깐 둔다
        os.kill(os.getpid(), 9)

print("설치 끝. 위의 빨간 ERROR 는 무시하면 됩니다. 다음 셀로.")


In [ ]:
# 전사 서버 — 접수하고(202) 뒤에서 돌리고, 앱이 몇 초마다 결과를 물어간다.
#
# 왜 비동기인가: 다 될 때까지 한 요청으로 기다리는 방식은 앱의 업로드
# 클라이언트(60초)와 Cloudflare 터널(약 100초)이 먼저 끊는다 — 실기기
# 타임아웃으로 재현된 사실이다.
#
# WhisperX 파이프라인은 단계로 돈다: 전사(배치) → 정렬 → (선택) 화자 분리.
# 배치 추론은 도중 진행률이 없어서, 단계(stage)로 알리고 전사가 끝난 시점에
# 거친 세그먼트를 통째로 조회 응답(?since 증분)에 싣는다 — 세션이 도중에
# 회수돼도 앱은 받은 데까지 저장한다. 정렬·화자는 완성본(result)에 실린다.
import os
import threading
import tempfile
import traceback
import uuid

from fastapi import FastAPI, File, Form, UploadFile
from fastapi.responses import HTMLResponse, JSONResponse


def build_app(run_pipeline, secret: str, log=print) -> FastAPI:
    # log: 백그라운드 스레드의 print 는 셀이 끝난 뒤에는 콜랩 화면에 안
    # 보인다. 실행 셀이 큐를 비우며 대신 찍도록 콜백으로 받는다.
    app = FastAPI()
    jobs = {}
    gpu_lock = threading.Lock()  # GPU 는 하나 — 작업을 줄 세운다.

    def run_job(job_id, path, language, requested_model, diarize, hf_token):
        job = jobs[job_id]
        try:
            with gpu_lock:
                run_pipeline(job, path, language, requested_model, diarize, hf_token)
            log(
                f"전사 끝({job_id[:8]}): {len(job['segments'])}문장"
                f" / {round(job['result']['duration'])}초 · 모델 {job.get('model')}"
            )
        except Exception:
            trace = traceback.format_exc()
            log(trace)
            job["error"] = trace[-1500:]
            job["status"] = "error"
        finally:
            os.unlink(path)


    # 비밀 주소의 첫 화면 — QR 로 들어온 폰이 버튼 한 번으로 앱과 연결된다.
    # 주소는 페이지 스스로 알아낸다(location) — 서버는 자기 공개 주소를 모른다.
    landing = """<!doctype html><html lang="ko"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>NSR 전사 서버</title>
<style>body{font-family:sans-serif;background:#131312;color:#eee;display:flex;min-height:100vh;align-items:center;justify-content:center;margin:0}
.card{max-width:340px;padding:28px;text-align:center}
.ok{color:#7fc8a9;font-weight:700;font-size:15px}
a.btn{display:block;margin-top:20px;padding:16px;border-radius:14px;background:#2f6b58;color:#fff;text-decoration:none;font-weight:700;font-size:17px}
p{line-height:1.5;font-size:14px;color:#bbb}
.ep{word-break:break-all;font-size:12px;color:#777}</style></head><body><div class="card">
<div class="ok">&#9679; NSR 전사 서버 살아 있음</div>
<p>이 폰에 NSR 앱이 있으면 아래 버튼 하나면 됩니다. 서버 주소가 앱에 자동 저장됩니다.</p>
<a class="btn" id="b" href="#">NSR 앱에 연결</a>
<p>버튼이 안 되면 아래 주소를 복사해 앱의 설정 &#8594; 전사 주소 칸에 붙여넣으십시오.</p>
<p class="ep" id="ep"></p></div>
<script>var ep=location.origin+location.pathname.replace(/\/+$/,"");
document.getElementById("b").href="nsr://connect?endpoint="+encodeURIComponent(ep);
document.getElementById("ep").textContent=ep;</script></body></html>"""

    @app.get(f"/{secret}")
    @app.get(f"/{secret}/")
    def home():
        return HTMLResponse(landing)

    @app.get(f"/{secret}/health")
    def health():
        return {"status": "ok"}

    @app.post(f"/{secret}/v1/audio/transcriptions")
    async def transcribe(
        file: UploadFile = File(...),
        language: str = Form("ko"),
        temperature: float = Form(0.0),
        prompt: str = Form(""),  # WhisperX 배치 경로는 프롬프트를 못 받는다 — 교정은 앱이 한다.
        response_format: str = Form("verbose_json"),
        model_name: str = Form("", alias="model"),  # 앱의 모델 선택 — 비면 세션 기본값
        diarize: str = Form(""),      # "1" 이면 화자 분리
        hf_token: str = Form(""),     # 앱의 설정에 저장된 허깅페이스 토큰 (화자 분리용)
    ):
        suffix = os.path.splitext(file.filename or "audio.m4a")[1] or ".m4a"
        with tempfile.NamedTemporaryFile(suffix=suffix, delete=False) as f:
            f.write(await file.read())
            path = f.name
        job_id = uuid.uuid4().hex
        jobs[job_id] = {"status": "queued", "progress": 0.0, "stage": "queued",
                        "segments": [], "model": None}
        threading.Thread(
            target=run_job,
            args=(job_id, path, language, model_name, diarize == "1", hf_token),
            daemon=True,
        ).start()
        log(
            f"전사 접수({job_id[:8]}): {file.filename}"
            + (
                f" · 앱이 고른 모델: {model_name}"
                if model_name
                else " · 앱이 모델을 안 보냈습니다 → 이 서버의 기본값을 씁니다"
            )
            + (" · 화자 분리" if diarize == "1" else "")
        )
        return JSONResponse(status_code=202, content={"job_id": job_id, "status": "queued"})

    @app.get(f"/{secret}/v1/audio/transcriptions/{{job_id}}")
    def job_status(job_id: str, since: int = 0):
        job = jobs.get(job_id)
        if job is None:
            return JSONResponse(
                status_code=404,
                content={"error": "모르는 작업입니다. 콜랩 세션이 재시작됐으면 전사를 다시 시작하십시오."},
            )
        if job["status"] == "done":
            return {"status": "done", "result": job["result"]}
        if job["status"] == "error":
            return {"status": "error", "error": job["error"]}
        done = job["segments"]
        return {
            "status": job["status"],
            "progress": round(job.get("progress", 0.0), 3),
            "stage": job.get("stage"),
            # 앱이 고른 모델과 서버가 실제로 실은 모델이 같은지 눈으로 확인할 수 있게.
            "model": job.get("model"),
            # since 이후의 새 세그먼트만 — 3초마다 물어도 응답이 가볍다.
            "segments": done[max(since, 0):],
            "next": len(done),
        }

    return app


In [ ]:
# 모델을 싣고 서버·터널을 띄운다. 마지막에 나오는 주소를 앱에 넣으면 된다.
import ctypes
import gc
import glob
import inspect
import os
import re
import secrets
import subprocess
import tarfile
import threading
import time
import urllib.parse
import urllib.request
import base64
import shutil
from io import BytesIO

import numpy as np
import psutil

# cuDNN/cuBLAS 를 먼저 손으로 적재한다. 콜랩 기본 환경의 판과 어긋나면
# 첫 전사에서 파이썬이 통째로 죽는데("kernel restarted", 추적도 안 남는다),
# 방금 설치한 판을 절대 경로로 미리 올려 두면 이후 탐색이 이쪽을 쓴다.
for pattern in (
    "/usr/local/lib/python3*/dist-packages/nvidia/cublas/lib/libcublas*.so*",
    "/usr/local/lib/python3*/dist-packages/nvidia/cudnn/lib/libcudnn*.so*",
):
    for lib in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib)
        except OSError:
            pass

import ctranslate2
import uvicorn
import whisperx
from huggingface_hub import snapshot_download

PORT = 8000
gpu = ctranslate2.get_cuda_device_count() > 0
DEVICE = "cuda" if gpu else "cpu"
BATCH = 16 if gpu else 4
if not gpu:
    print("⚠ GPU 가 안 잡혔습니다. 런타임 → 런타임 유형 변경 → T4 GPU 를 고른 뒤")
    print("  '모두 실행'을 다시 하십시오. CPU 로도 되지만 몇 배 느립니다.")

# ── 화자 분리 토큰 — 앱이 전사 요청에 실어 보낸다 ────────────
# pyannote 모델은 무료지만 허깅페이스 계정 확인을 요구한다. 그 토큰은
# 앱의 설정 → 전사 → '화자 분리'에서 한 번 넣어 두면 전사 요청마다 함께
# 온다. 여기서는 아무것도 설정하지 않는다.
# (예전 판을 쓰던 사람의 콜랩 보안 비밀은 조용히 예비로만 읽는다.)
try:
    from google.colab import userdata
    HF_TOKEN = (userdata.get("HF_TOKEN") or "").strip()
except Exception:
    HF_TOKEN = ""

# ── 전사 모델 준비 — 앱이 고른 모델을 게으르게 싣고, 다르면 갈아끼운다 ──
# 지난 사고들을 여기서 계속 막는다.
#  · 401 사고: 없는 저장소에 익명 요청이 가면 허깅페이스는 404 대신 401 을
#    준다. token=False 로 못박아 낡은 토큰 개입도 잠갔다(공개 모델만 받는다).
#  · 램 사고: 모델을 갈아끼울 때 이전 모델을 먼저 내려놓고 받는다.
#
# 앱이 모델을 안 보냈을 때만 쓰는 기본값. 보통은 앱의 '전사 모델' 목록이 정한다.
DEFAULT_MODEL_ID = "nsr-korean-medium"

NSR_ID = "nsr-korean-medium"
NSR_TURBO_ID = "nsr-korean-large-turbo"
MIRROR_BASE = "https://github.com/lulus-cat/NSR-project/releases/download/models/"
MIRRORS = {
    NSR_ID: (MIRROR_BASE + "ct2-korean-medium-1273h-fp16.tar.gz",
             "/content/nsr-korean-medium", "한국어 Medium(대화 특화)"),
    NSR_TURBO_ID: (MIRROR_BASE + "ct2-korean-large-v3-turbo-fp16.tar.gz",
                   "/content/nsr-korean-large-turbo", "한국어 Large Turbo(fp16)"),
}


def canonical(mid):
    mid = (mid or "").strip()
    if not mid:
        return None
    if mid in MIRRORS:
        return mid
    if "Large Turbo" in mid or "large-turbo" in mid:
        return NSR_TURBO_ID
    if mid.startswith("NSR"):
        return NSR_ID
    return mid


def fetch_model_dir(mid):
    """모델 파일을 마련하고 (경로, compute_type) 을 돌려준다."""
    if mid in MIRRORS:
        url, model_dir, label = MIRRORS[mid]
        if not os.path.exists(os.path.join(model_dir, "model.bin")):
            say(f"{label}을 받는 중 — 약 1.4GB, 1~3분...")
            done = [-1]
            def _pct(count, block, total):
                pct = min(count * block * 100 // max(total, 1), 100)
                if pct // 10 > done[0]:
                    done[0] = pct // 10
                    say(f"  ...{pct}%")
            tar_path, _ = urllib.request.urlretrieve(url, reporthook=_pct)
            os.makedirs(model_dir, exist_ok=True)
            with tarfile.open(tar_path) as tar:
                tar.extractall(model_dir, filter="data")
            os.remove(tar_path)
        return model_dir, ("float16" if gpu else "int8")

    compute = "float16" if gpu else "int8"
    if "korean" in mid.lower():
        # float32 로 저장된 파인튜닝판 — 변환 없이 그대로("default") 싣는다.
        # 무료 콜랩 램(12.7GB)은 넘길 수 있다. 유료 고용량 램에서 쓰는 항목이다.
        compute = "default" if gpu else "int8"
        say("한국어 파인튜닝 float32 판: 램을 많이 씁니다 — 무료 콜랩이면 fp16 판을 쓰십시오.")
    say(f"모델 받는 중: {mid} (이 세션에서 처음 쓸 때만 내려받습니다)")
    try:
        model_dir = snapshot_download(
            mid,
            token=False,
            allow_patterns=["config.json", "preprocessor_config.json", "model.bin",
                            "tokenizer.json", "vocabulary.*"],
        )
    except Exception as e:
        raise RuntimeError(
            f"모델을 못 받았습니다: {mid} — id 철자를 확인하십시오"
            " (공개된 CT2 형식 모델이어야 합니다. 401 은 대부분 없는 저장소라는 뜻입니다)."
        ) from e
    return model_dir, compute


# 백그라운드 스레드의 print 는 셀이 끝나면 화면에 안 보인다. 큐에 쌓고
# 아래 상주 루프가 대신 찍는다 — 서버가 뜨기 전(초기 적재)에는 바로 찍는다.
events = []
serving = threading.Event()

def say(msg):
    if serving.is_set():
        events.append(msg)
    else:
        print(msg, flush=True)


_current = {"id": None, "model": None}
_model_lock = threading.Lock()


def get_model(requested=None):
    """앱이 요청한 모델의 WhisperX 배치 모델. 안 고르면 위 셀의 기본값."""
    mid = canonical(requested) or canonical(DEFAULT_MODEL_ID) or NSR_ID
    with _model_lock:
        if _current["id"] == mid:
            return _current["model"]
        if _current["model"] is not None:
            say(f"모델 교체: {_current['id']} → {mid}")
            _current["model"] = None
            _current["id"] = None
            gc.collect()  # 이전 모델의 램·VRAM 을 먼저 돌려받는다.
        model_dir, compute = fetch_model_dir(mid)
        say(f"모델 여는 중: {mid}")
        model = whisperx.load_model(model_dir, DEVICE, compute_type=compute, language="ko")
        _current["id"] = mid
        _current["model"] = model
        return model


# 정렬(단어 시각) 모델 — 공개 한국어 wav2vec2 라 토큰이 필요 없다.
_align = {"model": None, "meta": None}
_align_lock = threading.Lock()


def get_align():
    with _align_lock:
        if _align["model"] is None:
            say("정렬 모델 준비 중 (한국어 wav2vec2, 처음 한 번 1~2분)...")
            _align["model"], _align["meta"] = whisperx.load_align_model(
                language_code="ko", device=DEVICE
            )
        return _align["model"], _align["meta"]


# 미리 실어 두면 첫 전사가 그만큼 빠르다. 다만 여기서 실패해도 서버는 떠야
# 한다 — 시작이 통째로 막히면 주소조차 못 받는다. 첫 전사 때 다시 시도한다.
try:
    get_align()
except Exception as _e:
    print(f"  지금은 못 실었습니다({type(_e).__name__}) — 첫 전사 때 다시 받습니다.")

# 화자 분리 파이프라인 — 토큰이 있을 때만 게으르게 싣는다.
#
# whisperx 는 판마다 여기가 바뀐다. 최신판(3.8)에서 두 가지가 달라졌다.
#  · 토큰 인자 이름: use_auth_token → token (pyannote 4 를 따라간 것이다)
#  · 기본 모델: speaker-diarization-3.1 → speaker-diarization-community-1
# 이름을 코드에 박아 두면 다음 판에서 또 깨진다. 생성자 서명을 읽어 있는
# 이름을 쓰고, 모델은 최신 것부터 차례로 시도한다.
try:
    from whisperx.diarize import DiarizationPipeline
except Exception:  # 판에 따라 자리만 다르다
    DiarizationPipeline = whisperx.DiarizationPipeline

_DIAR_TOKEN_ARG = next(
    (
        name
        for name in ("token", "use_auth_token")
        if name in inspect.signature(DiarizationPipeline.__init__).parameters
    ),
    None,
)

# 최신 기본값을 먼저, 예전 판을 그 다음으로. 둘 다 허깅페이스 '동의'가 필요하다.
DIAR_MODELS = [
    "pyannote/speaker-diarization-community-1",
    "pyannote/speaker-diarization-3.1",
]
DIAR_CONSENT = (
    "화자 분리 모델을 못 실었습니다. 아래 세 쪽을 열어 'Agree'(동의)를 누르고,"
    " 앱에 넣은 토큰이 Read 권한인지 확인하십시오.\n"
    "  https://huggingface.co/pyannote/speaker-diarization-community-1\n"
    "  https://huggingface.co/pyannote/speaker-diarization-3.1\n"
    "  https://huggingface.co/pyannote/segmentation-3.0"
)

_diarizer = {"token": None, "pipe": None}
_diar_lock = threading.Lock()


def get_diarizer(token):
    with _diar_lock:
        if _diarizer["pipe"] is not None and _diarizer["token"] == token:
            return _diarizer["pipe"]
        say("화자 분리 모델 준비 중 (처음 한 번, 1~2분)...")
        failures = []
        for name in DIAR_MODELS:
            kwargs = {_DIAR_TOKEN_ARG: token} if _DIAR_TOKEN_ARG else {}
            try:
                pipe = DiarizationPipeline(model_name=name, device=DEVICE, **kwargs)
            except Exception as e:
                failures.append(f"{name} → {type(e).__name__}: {e}")
                continue
            say(f"화자 분리 모델: {name}")
            _diarizer["token"] = token
            _diarizer["pipe"] = pipe
            return pipe
        raise RuntimeError(DIAR_CONSENT + "\n\n실제 오류:\n  " + "\n  ".join(failures))


def run_pipeline(job, path, language, requested_model, diarize, req_token):
    """WhisperX 3단: 배치 전사 → 강제 정렬 → (선택) 화자 분리."""
    job["status"] = "processing"
    job["stage"] = "model"
    job["model"] = canonical(requested_model) or canonical(DEFAULT_MODEL_ID) or NSR_ID
    model = get_model(requested_model)

    job["stage"] = "transcribe"
    audio = whisperx.load_audio(path)
    duration = round(len(audio) / 16000, 2)
    result = model.transcribe(audio, batch_size=BATCH, language=language or "ko")
    coarse = result["segments"]
    # 배치 전사는 도중 진행률이 없다 — 끝난 시점에 통째로 실어, 이후 단계에서
    # 세션이 회수돼도 앱이 여기까지는 건지게 한다.
    job["segments"].extend(
        {"id": i, "start": round(float(s["start"]), 2), "end": round(float(s["end"]), 2),
         "text": s["text"]}
        for i, s in enumerate(coarse)
    )
    job["progress"] = 0.7

    job["stage"] = "align"
    align_model, align_meta = get_align()
    aligned = whisperx.align(coarse, align_model, align_meta, audio, DEVICE,
                             return_char_alignments=False)
    job["progress"] = 0.85

    token = (req_token or HF_TOKEN).strip()
    if diarize:
        if token:
            job["stage"] = "diarize"
            dia = get_diarizer(token)(audio)
            aligned = whisperx.assign_word_speakers(dia, aligned)
        else:
            say("화자 분리가 켜져 있지만 토큰이 없어 건너뜁니다 —"
                " 앱의 설정 → 전사 → 화자 분리에서 허깅페이스 토큰을 넣으십시오.")

    final = []
    for i, s in enumerate(aligned["segments"]):
        start = s.get("start"); end = s.get("end")
        if start is None or end is None:  # 정렬이 못 잡은 짧은 조각은 원래 시각을 쓴다
            src = coarse[min(i, len(coarse) - 1)]
            start, end = src["start"], src["end"]
        seg = {"id": i, "start": round(float(start), 2), "end": round(float(end), 2),
               "text": s["text"]}
        if s.get("speaker"):
            seg["speaker"] = s["speaker"]
        final.append(seg)

    # 조회용 목록을 완성본으로 갈아끼운다(같은 리스트 객체를 유지해야
    # ?since 증분이 계속 동작한다).
    job["segments"][:] = final
    job["progress"] = 1.0
    job["result"] = {
        "task": "transcribe",
        "language": language or "ko",
        "duration": duration,
        "text": "".join(s["text"] for s in final).strip(),
        "segments": final,
    }
    job["status"] = "done"


# 자가 시험: 주소를 내주기 전에 1초짜리 무음을 배치 전사해 본다.
# GPU 경로(cuDNN/cuBLAS)가 죽을 거라면 여기서 바로 죽어 원인이 이 셀에 보인다.
#
# 시험에는 제일 작은 모델(tiny, 약 75MB)을 쓴다. 예전에는 기본 모델을 실었는데,
# 그러면 세션을 다시 켤 때마다 앱이 고르지도 않은 1.4GB 를 내려받고, 화면에는
# 그 모델 이름이 찍혀 "고른 모델이 무시된다"로 보였다. 시험의 목적은 GPU 경로
# 확인이지 모델 준비가 아니다 — 앱이 고른 모델은 첫 전사 때 실린다.
print("자가 전사 시험 중 (시험 전용 소형 모델, 몇 초)...")
try:
    _probe = whisperx.load_model(
        "tiny", DEVICE, compute_type=("float16" if gpu else "int8"), language="ko"
    )
    _probe.transcribe(np.zeros(16000, dtype=np.float32), batch_size=4, language="ko")
    del _probe
    gc.collect()
    print("자가 전사 시험 통과 — 전사 경로 정상.")
except Exception as _e:
    print(f"자가 시험을 못 돌렸습니다({type(_e).__name__}) — 서버는 그대로 띄웁니다.")
print("전사 모델은 앱에서 고른 것을 첫 전사 때 싣습니다(내려받느라 몇 분 걸립니다).")
vm = psutil.virtual_memory()
print(f"메모리 {vm.used / 1e9:.1f} / {vm.total / 1e9:.1f} GB 사용 중 — 10GB 를 넘어가면 위험하다.")

secret = secrets.token_urlsafe(12)
app = build_app(run_pipeline, secret, log=say)
threading.Thread(
    target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"),
    daemon=True,
).start()

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
deadline = time.time() + 60
while time.time() < deadline and url is None:
    line = tunnel.stdout.readline()
    if not line and tunnel.poll() is not None:
        break
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line or "")
    if m:
        url = m.group(0)
if not url:
    raise RuntimeError("터널 주소를 못 받았습니다. 이 셀만 한 번 더 실행해 보십시오.")

full = f"{url}/{secret}"

# 주소 복붙을 없앤다: 같은 폰이면 버튼 한 번, 컴퓨터면 QR 한 번.
import qrcode
from IPython.display import HTML as _HTML
from IPython.display import display as _display

_buf = BytesIO()
qrcode.make(full).save(_buf, format="PNG")
_qr64 = base64.b64encode(_buf.getvalue()).decode()
_du = shutil.disk_usage("/")
_disk = (
    f"콜랩 디스크 {_du.used / 1e9:.0f}/{_du.total / 1e9:.0f} GB 사용 — 세션이 끝나면 통째로"
    " 초기화되는 임시 공간이라, 구글 드라이브 용량은 먹지 않습니다."
)
_display(_HTML(f"""
<div style="font-family:sans-serif;max-width:560px;line-height:1.55">
  <h2 style="margin:8px 0">연결 준비 완료</h2>
  <p style="margin:8px 0 4px"><b>방법 1 — 이 화면을 폰으로 보고 있다면:</b></p>
  <p style="margin:4px 0"><a href="nsr://connect?endpoint={urllib.parse.quote(full, safe='')}"
        style="display:inline-block;padding:14px 22px;border-radius:12px;background:#2f6b58;color:#fff;text-decoration:none;font-weight:700">
        NSR 앱에 연결 (주소 자동 저장)</a></p>
  <p style="margin:14px 0 4px"><b>방법 2 — 컴퓨터로 보고 있다면:</b> 폰 카메라로 아래 QR 을 찍고,
     열리는 페이지에서 'NSR 앱에 연결'을 누르십시오.</p>
  <img src="data:image/png;base64,{_qr64}" width="190" height="190" style="image-rendering:pixelated"/>
  <p style="margin:14px 0 4px"><b>방법 3 — 수동:</b> 이 주소를 복사해 앱의 설정 → 전사 주소 칸에 붙여넣기.</p>
  <p style="word-break:break-all;background:#f5f5f5;color:#222;padding:10px;border-radius:8px;font-family:monospace">{full}</p>
  <p style="color:#888;font-size:13px">{_disk}<br/>
  전사 모델·화자 분리·허깅페이스 토큰은 전부 <b>앱의 설정 → 전사</b>에서 정하고, 전사 요청에
  함께 실려 옵니다 — 이 노트에서 고칠 것은 없습니다. 이 셀은 계속 실행 중인 것이 정상이고,
  전사 접수/완료 로그가 아래에 찍힙니다. 탭을 닫으면 서버도 꺼집니다.</p>
</div>
"""))
serving.set()
while True:
    time.sleep(2)
    while events:
        print(events.pop(0), flush=True)
